In [2]:
import os
from pathlib import Path
import sys
sys.path.append('/Users/jaideepmanupati/Downloads')
import csv
import argparse
import pandas as pd
from pydub import AudioSegment
from audio import preprocess
from telugu_tts.datasets.telugu_speech import TeluguDataset

def get_audio_duration(file_path):
    audio = AudioSegment.from_wav(file_path)
    duration_seconds = len(audio) / 1000.0  # convert milliseconds to seconds
    return duration_seconds

def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument("--dataset", required=True, choices=['te_in_male'], help='dataset name')
    args = parser.parse_args()

    if args.dataset == 'te_in_male':
        dataset_name = 'te_in_male'
        current_script_path = Path(__file__).resolve().parent
        project_root = current_script_path.parent
        datasets_path = project_root / 'datasets'
        dataset_path = datasets_path / dataset_name

        tsv_file_path = datasets_path / f'{args.dataset}/{args.dataset}.tsv'

        if not tsv_file_path.is_file():
            print(f"File '{tsv_file_path}' does not exist.")
            sys.exit(1)  # Exit the script if the file is not found
        else:
            print(f"Using existing file '{tsv_file_path}'")

            tsv_df = pd.read_csv(tsv_file_path, sep='\t')

            if not dataset_path.is_dir():
                dataset_path.mkdir()

                metadata_csv_path = dataset_path / 'metadata.csv'
                with open(metadata_csv_path, 'w', newline='') as metadata_csv:
                    metadata_csv_writer = csv.writer(metadata_csv, delimiter='|')

                    total_duration_s = 0

                    # Print column names
                    for col in tsv_df.columns:
                        print(col)

                    # Modify this part based on your column names
                    for index, row in tsv_df.iterrows():
                        filename, text = row['Filename'], row['Text']
                        wav_path = datasets_path / 'wavs' / f'{filename}.wav'

                        duration_s = get_audio_duration(wav_path)

                        # For the purpose of this example, we assume the duration is 10 seconds
                        duration_s = 10
                        total_duration_s += duration_s

                        metadata_csv_writer.writerow([filename, text, text])

                    print(f"Total audio duration: {total_duration_s}s")

                    # pre-process
                    print("Pre-processing...")

                    telugu_speech = TeluguDataset([])  # Adjust based on your dataset structure

                    preprocess_output_folder = 'output_folder'  # Replace with the desired output folder
                    preprocess(dataset_path, telugu_speech, output_folder=preprocess_output_folder)

    else:
        print(f"Error: The specified dataset '{args.dataset}' is not supported.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'audio'